# 01_preprocessing.ipynb — HistoBreastNet (Persona 1) · v4

Produce DUE configurazioni dataset sotto `data/processed/`:
- **`cap380_1p86GB`** — cap per sottotipo (config di confronto)
- **`diversity_1p5GB`** — budget ~1,5 GB, diversity-preferring (config principale)

Ogni config: `metadata_subset.csv`, `subset_manifest.csv`, `image_wise_split.csv`, `patient_wise_split.csv`, `patient_wise_folds.csv` (k=5), `statistics.csv`, `config.json`. Tutti i CSV con `relative_path` e `dataset_config`. Persona 2 **consuma** questi file.

**Prerequisito:** `notebooks/dataset_reduction.py` deve essere la **v4** (Sez. 2 lo verifica).

## Sezione 0 — Setup e path

In [1]:
from google.colab import drive
from pathlib import Path
import sys, shutil, time
drive.mount('/content/drive')

PROJECT_ROOT   = Path('/content/drive/MyDrive/HistoBreastNet')
SRC_DIR        = PROJECT_ROOT/'notebooks'
DATA_PROCESSED = PROJECT_ROOT/'data'/'processed'
RANDOM_STATE   = 42

assert PROJECT_ROOT.exists(), f"PROJECT_ROOT non risolve: {PROJECT_ROOT} (controlla la scorciatoia)"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
print("PROJECT_ROOT OK:", PROJECT_ROOT)
print("data/processed :", DATA_PROCESSED)

Mounted at /content/drive
PROJECT_ROOT OK: /content/drive/MyDrive/HistoBreastNet
data/processed : /content/drive/MyDrive/HistoBreastNet/data/processed


## Sezione 1 — Dataset in locale (robusto alle disconnessioni)

Strategia: si lavora da uno **ZIP** su Drive. La prima cella:
1. se il dataset è già completo in locale → non fa nulla;
2. se lo **zip** esiste su Drive → lo estrae (veloce, 1–3 min);
3. se lo zip non esiste ancora → copia da `data/original/` in modo **resumable** (se il runtime si disconnette, **rilancia la cella**: riprende da dove era).

La seconda cella crea lo zip **una volta sola**, così le sessioni future partono in pochi minuti.

In [2]:
import subprocess, zipfile, shutil, time
from pathlib import Path

BREAKHIS_ORIG   = PROJECT_ROOT/'data'/'original'/'BreaKHis_v1'       # cartella su Drive
BREAKHIS_ZIP    = PROJECT_ROOT/'data'/'original'/'BreaKHis_v1.zip'   # zip su Drive (creato una volta)
BREAKHIS_LOCAL  = Path('/content/breakhis/BreaKHis_v1')
BREAKHIS_SOURCE = BREAKHIS_LOCAL

def n_local():
    return sum(1 for _ in BREAKHIS_LOCAL.rglob('*.png')) if BREAKHIS_LOCAL.exists() else 0

n = n_local()
if n == 7909:
    print("Dataset già completo in locale.")
elif BREAKHIS_ZIP.exists():
    shutil.rmtree('/content/breakhis', ignore_errors=True)
    BREAKHIS_LOCAL.mkdir(parents=True, exist_ok=True)
    t0 = time.time(); print("Estrazione dallo zip su Drive...")
    with zipfile.ZipFile(BREAKHIS_ZIP) as z:
        z.extractall(BREAKHIS_LOCAL)
    n = n_local(); print(f"Estratto in {time.time()-t0:.0f}s")
else:
    BREAKHIS_LOCAL.mkdir(parents=True, exist_ok=True)
    print("Zip non presente: copio da data/original (resumable).")
    print("Se il runtime si disconnette, RILANCIA questa cella: riprende da dove era.")
    t0 = time.time()
    subprocess.run(['rsync','-a','--ignore-existing',
                    str(BREAKHIS_ORIG)+'/', str(BREAKHIS_LOCAL)+'/'])
    n = n_local(); print(f"{time.time()-t0:.0f}s")

print("Immagini .png:", n, "-> OK ✓" if n==7909 else f"-> [INCOMPLETO {n}] rilancia la cella")

assert n == 7909, f"Dataset locale incompleto: trovate {n} immagini, attese 7909. Rilancia la cella prima di proseguire."

# Normalizza la root effettiva: deve contenere direttamente histology_slides/
if (BREAKHIS_LOCAL / "histology_slides").exists():
    BREAKHIS_SOURCE = BREAKHIS_LOCAL
elif (BREAKHIS_LOCAL / "BreaKHis_v1" / "histology_slides").exists():
    BREAKHIS_SOURCE = BREAKHIS_LOCAL / "BreaKHis_v1"
else:
    raise FileNotFoundError(
        f"Non trovo histology_slides dentro {BREAKHIS_LOCAL}. "
        "Controlla la struttura dello zip estratto."
    )
print("BREAKHIS_SOURCE effettivo:", BREAKHIS_SOURCE)

Estrazione dallo zip su Drive...
Estratto in 102s
Immagini .png: 7909 -> OK ✓
BREAKHIS_SOURCE effettivo: /content/breakhis/BreaKHis_v1


In [3]:
# UNA TANTUM: crea lo zip su Drive (dalla copia locale, veloce) per le sessioni future.
# Eseguila solo quando il conteggio sopra è 7909. Non fa nulla se lo zip esiste già.
if n_local() == 7909 and not BREAKHIS_ZIP.exists():
    t0 = time.time(); print("Creo lo zip su Drive dalla copia locale...")
    shutil.make_archive(str(BREAKHIS_ZIP.with_suffix('')), 'zip', BREAKHIS_LOCAL)
    print(f"Zip creato in {time.time()-t0:.0f}s ->", BREAKHIS_ZIP)
elif BREAKHIS_ZIP.exists():
    print("Zip già presente su Drive:", BREAKHIS_ZIP)
else:
    print("Copia locale non ancora completa (serve 7909): completa la cella precedente prima.")

Zip già presente su Drive: /content/drive/MyDrive/HistoBreastNet/data/original/BreaKHis_v1.zip


## Sezione 2 — Import modulo

In [4]:
module_drive = SRC_DIR/'dataset_reduction.py'
assert module_drive.exists(), "dataset_reduction.py non è in notebooks/. Caricalo prima."

sys.path.insert(0, str(SRC_DIR))
import importlib, dataset_reduction as dr; importlib.reload(dr)

<module 'dataset_reduction' from '/content/drive/MyDrive/HistoBreastNet/notebooks/dataset_reduction.py'>

## Sezione 3 — Metadata completo (config-independent)

In [ ]:
metadata_full = dr.extract_metadata(BREAKHIS_SOURCE)
n = metadata_full['patient_id'].nunique()
metadata_full.to_csv(DATA_PROCESSED/'metadata_full.csv', index=False)
print("Immagini:", len(metadata_full), "| pazienti:", n, "-> OK 82 ✓" if n==82 else f"-> [nota] {n}")
print("Colonne (nota relative_path):", list(metadata_full.columns))

## Sezione 4 — Config A: `cap380_1p86GB` (confronto)

In [ ]:
sel_a, sub_a, man_a = dr.build_subset(metadata_full, images_per_subtype=380, random_state=RANDOM_STATE)
iw_a, pw_a = dr.make_splits(sub_a, random_state=RANDOM_STATE)
kf_a = dr.make_kfold_patient_splits(sub_a, k=5, random_state=RANDOM_STATE)

dir_a, cfg_a = dr.write_config_bundle(
    sub_a, man_a, iw_a, pw_a, kf_a, DATA_PROCESSED/'cap380_1p86GB',
    dataset_config='cap380_1p86GB', selection_strategy='subtype_cap_patient_level',
    random_state=RANDOM_STATE, images_per_subtype_cap=380)

print("Config A ->", dir_a)
print(f"  pazienti={cfg_a['n_patients']} immagini={cfg_a['n_images']} "
      f"size={cfg_a['actual_size_gib']} GiB ({cfg_a['actual_size_gb_decimal']} GB dec)")
print("  classe (pazienti):", cfg_a['n_patients_benign'], "benign /", cfg_a['n_patients_malignant'], "malignant")
print("  fold validi:", cfg_a['folds_valid'])

## Sezione 5 — Config B: `diversity_1p5GB` (principale)

In [ ]:
sel_b, sub_b, man_b = dr.build_subset_budget(metadata_full, target_gb=1.5, random_state=RANDOM_STATE)
iw_b, pw_b = dr.make_splits(sub_b, random_state=RANDOM_STATE)
kf_b = dr.make_kfold_patient_splits(sub_b, k=5, random_state=RANDOM_STATE)

dir_b, cfg_b = dr.write_config_bundle(
    sub_b, man_b, iw_b, pw_b, kf_b, DATA_PROCESSED/'diversity_1p5GB',
    dataset_config='diversity_1p5GB', selection_strategy='diversity_preferring_patient_level',
    random_state=RANDOM_STATE, target_size_gb=1.5)

print("Config B ->", dir_b)
print(f"  pazienti={cfg_b['n_patients']} immagini={cfg_b['n_images']} "
      f"size={cfg_b['actual_size_gib']} GiB ({cfg_b['actual_size_gb_decimal']} GB dec)")
print("  per sottotipo:", cfg_b['subtypes'])
print("  classe (pazienti):", cfg_b['n_patients_benign'], "benign /", cfg_b['n_patients_malignant'], "malignant")
print("  test patient-wise (n pazienti):", pw_b[pw_b.split=='test']['patient_id'].nunique())
print("  fold validi:", cfg_b['folds_valid'])

## Sezione 6 — Verifica finale (file + anti-leakage per fold)

In [ ]:
import pandas as pd
for cfg_dir in sorted(DATA_PROCESSED.glob('*/')):
    if not (cfg_dir/'config.json').exists():
        continue
    print(f"== {cfg_dir.name} ==")
    for f in sorted(cfg_dir.glob('*')):
        print(f"   • {f.name:26s} {f.stat().st_size/1024:8.1f} KB")
    kf = pd.read_csv(cfg_dir/'patient_wise_folds.csv')
    print("   fold validi (zero overlap + test una volta):", dr.verify_folds(kf, k=kf['fold'].nunique()), "\n")

## Sezione 7 — Copia fisica del subset (`images/`)

Crea una copia fisica delle **sole** immagini del subset `diversity_1p5GB` sotto
`data/processed/diversity_1p5GB/images/`, mantenendo la struttura relativa che parte da
`histology_slides/...`. Serve a rendere il training più veloce (niente lettura di migliaia
di file sparsi da `data/original/`) e permette a Persona 2 di risolvere i path come
`DATASET_ROOT / relative_path` con `DATASET_ROOT = .../images`.

- **Non tocca mai** `data/original/BreaKHis_v1` → solo lettura, nessun move/delete/rename.
- **Idempotente / resumable**: se il file destinazione esiste con la **stessa** dimensione non viene ricopiato; se ha dimensione **diversa** viene sovrascritto. In caso di disconnessione, **rilancia la cella**: riprende da dove era.
- Gli split restano gestiti dai CSV → **nessuna** sottocartella `train/`/`val/`/`test/`.
- `relative_path` resta invariata (usata dagli altri notebook). Viene solo, opzionalmente, **aggiunta** la colonna `subset_relative_path` (identica a `relative_path`).

In [5]:
import pandas as pd
from pathlib import Path
import shutil, time

# ---------------- Config Sezione 7 ----------------
DATASET_CONFIG         = "diversity_1p5GB"
CREATE_PHYSICAL_SUBSET = True          # copia fisica del subset
ADD_SUBSET_RELPATH_COL = True          # aggiunge 'subset_relative_path' (== relative_path), senza sostituirla

PROCESSED_DIR          = DATA_PROCESSED / DATASET_CONFIG
METADATA_SUBSET_CSV    = PROCESSED_DIR / 'metadata_subset.csv'
SUBSET_IMAGES_DIR      = PROCESSED_DIR / 'images'
ORIGINAL_DATASET_ROOT  = PROJECT_ROOT / 'data' / 'original' / 'BreaKHis_v1'   # MAI modificato

# Sorgente di LETTURA: preferisci la copia locale già estratta in Sezione 1 (molto più veloce),
# altrimenti fallback sull'originale su Drive. In entrambi i casi è sola lettura.
try:
    _local_root = BREAKHIS_SOURCE                      # definito in Sezione 1 se disponibile
except NameError:
    _local_root = Path('/content/breakhis/BreaKHis_v1')
SOURCE_ROOT = _local_root if _local_root.exists() else ORIGINAL_DATASET_ROOT
assert SOURCE_ROOT.exists(), f"Nessuna sorgente valida: {_local_root} oppure {ORIGINAL_DATASET_ROOT}"

print("Sorgente lettura :", SOURCE_ROOT, "(locale)" if SOURCE_ROOT == _local_root else "(Drive)")
print("Destinazione     :", SUBSET_IMAGES_DIR)

if not CREATE_PHYSICAL_SUBSET:
    print("CREATE_PHYSICAL_SUBSET=False -> salto la copia.")
else:
    assert METADATA_SUBSET_CSV.exists(), f"Manca {METADATA_SUBSET_CSV} (esegui la Sezione 5 prima)."
    df = pd.read_csv(METADATA_SUBSET_CSV)
    assert 'relative_path' in df.columns, "Colonna 'relative_path' assente in metadata_subset.csv"

    SUBSET_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
    n_csv = len(df)
    n_present = n_copied = n_missing = 0
    missing = []
    t0 = time.time()
    for i, rel in enumerate(df['relative_path'].astype(str), 1):
        src = SOURCE_ROOT / rel
        if not src.exists() and SOURCE_ROOT != ORIGINAL_DATASET_ROOT:
            src = ORIGINAL_DATASET_ROOT / rel      # fallback: copia locale incompleta -> leggi da Drive
        dst = SUBSET_IMAGES_DIR / rel
        if not src.exists():
            n_missing += 1; missing.append(rel); continue
        s_size = src.stat().st_size
        if dst.exists() and dst.stat().st_size == s_size:
            n_present += 1                              # già copiato, stessa dimensione -> skip
        else:
            dst.parent.mkdir(parents=True, exist_ok=True)   # crea le sottocartelle necessarie
            shutil.copy2(src, dst)                     # copia (sovrascrive se dimensione diversa)
            n_copied += 1
        if i % 500 == 0:
            print(f"  ...{i}/{n_csv}  (presenti={n_present} copiate={n_copied} mancanti={n_missing})")
    dt = time.time() - t0

    # Colonna opzionale: NON sostituisce relative_path
    if ADD_SUBSET_RELPATH_COL:
        if 'subset_relative_path' not in df.columns:
            df['subset_relative_path'] = df['relative_path']
            df.to_csv(METADATA_SUBSET_CSV, index=False)
            print("Aggiunta colonna 'subset_relative_path' a metadata_subset.csv")
        else:
            print("Colonna 'subset_relative_path' già presente: nessuna modifica.")

    total_bytes = sum(p.stat().st_size for p in SUBSET_IMAGES_DIR.rglob('*.png'))
    print("\n===== RIEPILOGO COPIA =====")
    print(f"immagini nel CSV     : {n_csv}")
    print(f"gia' presenti (skip) : {n_present}")
    print(f"copiate ora          : {n_copied}")
    print(f"mancanti (sorgente)  : {n_missing}")
    print(f"dimensione images/   : {total_bytes/1024**3:.3f} GiB ({total_bytes/1e9:.3f} GB dec)")
    print(f"tempo totale         : {dt:.0f}s")
    if missing:
        print("Esempi mancanti:", missing[:10])

Sorgente lettura : /content/breakhis/BreaKHis_v1 (locale)
Destinazione     : /content/drive/MyDrive/HistoBreastNet/data/processed/diversity_1p5GB/images
  ...500/2838  (presenti=0 copiate=500 mancanti=0)
  ...1000/2838  (presenti=0 copiate=1000 mancanti=0)
  ...1500/2838  (presenti=0 copiate=1500 mancanti=0)
  ...2000/2838  (presenti=0 copiate=2000 mancanti=0)
  ...2500/2838  (presenti=0 copiate=2500 mancanti=0)
Aggiunta colonna 'subset_relative_path' a metadata_subset.csv

===== RIEPILOGO COPIA =====
immagini nel CSV     : 2838
gia' presenti (skip) : 0
copiate ora          : 2838
mancanti (sorgente)  : 0
dimensione images/   : 1.483 GiB (1.592 GB dec)
tempo totale         : 76s


### Validazioni copia fisica

Controlli bloccanti: conteggio immagini, presenza di ogni `relative_path` sotto `images/`,
distribuzioni identiche a `metadata_subset.csv` (`label`, `binary_label`, `subtype`,
`magnification`, `patient_id`).

In [6]:
import pandas as pd
df = pd.read_csv(METADATA_SUBSET_CSV)
rels = df['relative_path'].astype(str)

# 1) numero .png nella cartella == numero righe CSV
n_png_folder = sum(1 for _ in SUBSET_IMAGES_DIR.rglob('*.png'))
assert n_png_folder == len(df), f"Conteggio diverso: folder={n_png_folder} csv={len(df)}"

# 2) ogni relative_path del CSV esiste sotto images/ (zero mancanti)
missing_dst = [r for r in rels if not (SUBSET_IMAGES_DIR / r).exists()]
assert not missing_dst, f"Mancano {len(missing_dst)} file sotto images/, es: {missing_dst[:5]}"

# 3) distribuzioni invariate rispetto a metadata_subset.csv
present = df[df['relative_path'].map(lambda r: (SUBSET_IMAGES_DIR / r).exists())]
for col in ['label', 'binary_label', 'subtype', 'magnification', 'patient_id']:
    a = df[col].value_counts().sort_index()
    b = present[col].value_counts().sort_index()
    assert a.equals(b), f"Distribuzione '{col}' cambiata"

# 4) relative_path resta invariata; subset_relative_path (se presente) è identica
assert 'relative_path' in df.columns, "relative_path mancante!"
if 'subset_relative_path' in df.columns:
    assert (df['subset_relative_path'].astype(str) == rels).all(), "subset_relative_path != relative_path"

print("Validazioni OK",
      "| immagini:", n_png_folder,
      "| pazienti:", df['patient_id'].nunique(),
      "| subtypes:", df['subtype'].nunique(),
      "| magnifications:", df['magnification'].nunique())

Validazioni OK | immagini: 2838 | pazienti: 33 | subtypes: 8 | magnifications: 4


### Zip opzionale del subset

Crea `data/processed/diversity_1p5GB/images.zip` con struttura `images/histology_slides/...`,
così Persona 2 può estrarlo in `/content/` e usare `DATASET_ROOT = <estratto>/images`.

Controllato da `CREATE_IMAGES_ZIP` (default `False`).
**Nota:** aggiungi `data/processed/*/images/` e `data/processed/*/images.zip` al `.gitignore` — non vanno committati.

In [8]:
# Zip opzionale (estrazione veloce lato Persona 2). Default: disattivato.
CREATE_IMAGES_ZIP = True
SUBSET_IMAGES_ZIP = PROCESSED_DIR / 'images.zip'

if CREATE_IMAGES_ZIP:
    import shutil, time
    t0 = time.time()
    print("Creo/aggiorno lo zip del subset...")
    # root_dir=PROCESSED_DIR + base_dir='images' -> gli entry hanno prefisso 'images/...'
    shutil.make_archive(str(SUBSET_IMAGES_ZIP.with_suffix('')), 'zip',
                        root_dir=str(PROCESSED_DIR), base_dir='images')
    print(f"Zip creato in {time.time()-t0:.0f}s ->", SUBSET_IMAGES_ZIP,
          f"({SUBSET_IMAGES_ZIP.stat().st_size/1024**3:.3f} GiB)")
else:
    print("CREATE_IMAGES_ZIP=False -> nessuno zip creato.")

Creo/aggiorno lo zip del subset...
Zip creato in 120s -> /content/drive/MyDrive/HistoBreastNet/data/processed/diversity_1p5GB/images.zip (1.483 GiB)
